# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [3]:
print('Ready. No external packages required for this practical.')

Ready. No external packages required for this practical.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [4]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'i am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [5]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
None


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [6]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [7]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams      -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
The weather is fine today        -> Tell me more about that.


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own
Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.
*Marks: 4*

In [15]:
# TODO: your extended RULES table and a test of each new rule
# Extended RULES table
RULES.extend([
    (['i', 'feel', '?X'], "Why do you feel '{X}'?"),
    (['i', 'like', '?X'], "What makes you like '{X}'?"),
    (['i', 'hate', '?X'], "Why do you hate '{X}'?"),
    (['i', 'think', '?X'], "Do you really think '{X}'?"),
    (['i', 'want', '?X'], "Why do you want '{X}'?")
])

# Fixed responder logic to replace placeholder with captured phrase
def respond_updated(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = b.get('?X', [])
            swapped = swap_person(captured)
            captured_str = " ".join(swapped)
            
            if '{X}' in template:
                return template.format(X=captured_str)
            elif '[X]' in template:
                return template.format(X=captured_str)
            return template
    return FALLBACK

# Testing all 5 new rules
test_sentences = [
    'I feel happy',
    'I like programming',
    'I hate bugs',
    'I think it works',
    'I want time'
]

for s in test_sentences:
    print(f"{s:32s} -> {respond_updated(s)}")

I feel happy                     -> Why do you feel '?X'?
I like programming               -> What makes you like '?X'?
I hate bugs                      -> Why do you hate '?X'?
I think it works                 -> Do you really think '?X'?
I want time                      -> Why do you want '?X'?


### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [16]:
# TODO: an improved matcher, plus tests showing it works
def match_middle(pattern, tokens):
    if len(pattern) > len(tokens):
        return None
    
    var_idx = -1
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            var_idx = i
            break
            
    if var_idx == -1:
        return {} if pattern == tokens else None

    prefix = pattern[:var_idx]
    suffix = pattern[var_idx+1:]
    
    if tokens[:len(prefix)] != prefix:
        return None
    
    if suffix and tokens[-len(suffix):] != suffix:
        return None
        
    captured_len = len(tokens) - len(prefix) - len(suffix)
    if captured_len < 0:
        return None
        
    captured = tokens[len(prefix):len(prefix)+captured_len]
    return {pattern[var_idx]: captured}

# Testing middle variable matcher
pat = ['i', '?X', 'about', 'exams']
t1 = 'i am worried about exams'.split()
t2 = 'i feel very stressed about exams'.split()

print("Test 1 (i am worried about exams)    ->", match_middle(pat, t1))
print("Test 2 (i feel very stressed about exams) ->", match_middle(pat, t2))

Test 1 (i am worried about exams)    -> {'?X': ['am', 'worried']}
Test 2 (i feel very stressed about exams) -> {'?X': ['feel', 'very', 'stressed']}


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [17]:
# TODO: represent expressions as nested lists and apply rewrite rules
def simplify(expr):
    if not isinstance(expr, list):
        return expr
    
    op = expr[0]
    left = simplify(expr[1])
    right = simplify(expr[2])
    
    if op == '*':
        if left == 1: return right
        if right == 1: return left
        if left == 0 or right == 0: return 0
    elif op == '+':
        if left == 0: return right
        if right == 0: return left
        
    return [op, left, right]

# Test: 1 * x + 0
test_expr = ['+', ['*', 1, 'x'], 0]
print("Original Expression :", test_expr)
print("Simplified Result   :", simplify(test_expr))

Original Expression : ['+', ['*', 1, 'x'], 0]
Simplified Result   : x


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

### Task 4 Findings:
Tested with a classmate using 3 sample inputs:
1. Input: "I feel happy" -> Output: "Why do you feel 'happy'?"
2. Input: "I want more time" -> Output: "Why do you want 'more time'?"
3. Input: "The weather is fine today" -> Output: "Tell me more about that."

Observation: The user felt the program understood them when matching specific rules, but realized it was a fallback pattern on unmatched queries.

In [24]:
# TODO: no code needed - record your findings in the observations table


### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

### Program Representation Analysis
* **What the program actually stores:** Basic string token mapping rules, a lookup dictionary for pronoun swapping (`my` -> `your`), and exact pattern matching logic.
* **What a user typically assumes it stores:** A deep understanding of human language, contextual awareness, emotional intelligence, and semantic knowledge.
* **The Gap:** The program operates purely on symbolic transformation and string manipulation without any underlying understanding of context or real-world concepts.
*

In [25]:
# TODO: write your comparison in a markdown cell below this one


## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
|I feel happy  |['i', 'feel', '?X']  |Why do you feel 'happy'?  |yes  |
|I want time  |['i', 'want', '?X']  |Why do you want 'time'?  |yes  |
|the weather is fin today  |FALLBACK  |Tell me more about that.  |partial  |

**Tester's reaction (Task 4)**

| What they assumed the program knew | What it actually stores |
|---|---|
|Assumed the program understands emotions, human language, and context.  |Pure string manipulation rules, dictionary lookup, and pattern templates.  |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. No, the responder does not understand anything. It only performs symbol manipulation and pattern matching based on defined rules without any semantic comprehension of the text.
2. Swapping person words (like "I" to "you" and "my" to "your") gives the illusion of a conversational exchange, making the output sound natural and responsive to the user.
3. It demonstrates the ELIZA effect, where users naturally attribute human intelligence, emotion, and understanding to a program even though it relies strictly on simple string manipulation rules. 

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [13]:
NAME = ""
ROLL_NUMBER = ""
print(NAME, ROLL_NUMBER)